In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,SystemMessage,ToolMessage
from langchain_core.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [15]:
load_dotenv(override=True)
parser = StrOutputParser()
llm = ChatOpenAI(model="gpt-5.4-mini")

In [16]:
def get_chat_history(session_id:str):
    return SQLChatMessageHistory(session_id=session_id,connection="sqlite:///chat_history.db")

In [17]:
system_message = SystemMessage("You are a helpful assistant")
user_message = HumanMessage("{input}")

system_prompt = SystemMessagePromptTemplate.from_template(system_message.content)
user_prompt = HumanMessagePromptTemplate.from_template(user_message.content)

messages = [system_message,user_message,MessagesPlaceholder(variable_name='history')]

prompt_template = ChatPromptTemplate(messages)

chain = prompt_template | llm | parser

chat_history_runnable = RunnableWithMessageHistory(chain,get_chat_history,input_messages_key='input',history_messages_key='history')

In [18]:
def chat(session_id,input):
    output = chat_history_runnable.invoke({"input":input},config={"configurable":{"session_id":session_id}})
    return output

In [19]:
user_id = "test123333"
chat(user_id,"my name is ganesh vadlamuri")

'Hi Ganesh Vadlamuri! Nice to meet you. How can I help you?'

In [20]:
chat(user_id,"what is my name?")

'Hi Ganesh Vadlamuri! Nice to meet you. How can I help you?'